# Ćwiczenie 6: Cloud Computing — BigQuery, Dataproc i Spark w chmurze GCP

**Kurs:** Python w Zastosowaniach Big Data — PJATK  
**Data:** 31.05.2026  
**Czas realizacji:** 90 minut  
**Poziom trudności:** ⭐⭐ (Intermediate)  

---

## Cele kształcenia

Po ukończeniu tego notebooka będziesz potrafił/a:

1. **Łączyć się z BigQuery** z poziomu Pythona i odpytywać publiczne zbiory danych
2. **Eksplorować duże datasety** w BigQuery bez konieczności pobierania ich na lokalną maszynę
3. **Pisać zaawansowane zapytania SQL** bezpośrednio w chmurze
4. **Wizualizować wyniki** z BigQuery w pandas i matplotlib
5. **Tworzyć klastry Dataproc** i rozumieć ich konfigurację
6. **Submitować joby PySpark** do klastra Dataproc
7. **Szacować koszty** usług chmurowych (BigQuery, Dataproc)

## Wymagania wstępne

- Podstawy SQL (SELECT, WHERE, GROUP BY, ORDER BY)
- Znajomość pandas (DataFrame, wizualizacja)
- Podstawy PySpark (z poprzednich ćwiczeń)
- Konto Google (do autoryzacji w Colab)

## Struktura laboratorium

| Część | Ćwiczenia | Opis |
|-------|-----------|------|
| **Część A** | 1–4 | BigQuery z poziomu Pythona (działa w Colab, free tier) |
| **Część B** | 5–7 | Dataproc — walkthrough i symulacja (gcloud CLI) |
| **Bonus** | 8 | Cloud Skills Boost — darmowe laby Google |

---

> **Wskazówka:** Uruchamiaj komórki po kolei od góry do dołu za pomocą `Shift+Enter`.  
> Jeśli utkniesz — kliknij komórkę z podpowiedzią lub zapytaj prowadzącego!

In [ ]:
# @title Konfiguracja środowiska — uruchom tę komórkę jako pierwszą!

!pip install -q google-cloud-bigquery db-dtypes pandas matplotlib

# Autoryzacja w Google Cloud (Colab poprosi o logowanie)
from google.colab import auth
auth.authenticate_user()

# Biblioteki
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Ustawienia wyświetlania
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Pakiety zainstalowane i autoryzacja zakończona!')
print('Środowisko gotowe — zaczynamy pracę z chmurą Google!')

## Wprowadzenie — Serverless Analytics w GCP

Wyobraź sobie, że masz **petabajty danych** i chcesz je przeanalizować, ale NIE chcesz:
- kupować serwerów
- instalować baz danych
- zarządzać infrastrukturą

Tutaj wchodzi **Google Cloud Platform** z usługami, które robią to za Ciebie!

### Kluczowe usługi GCP dla Big Data

| Usługa | Typ | Do czego służy | Odpowiednik |
|--------|-----|----------------|-------------|
| **BigQuery** | Serverless data warehouse | Zapytania SQL na petabajtach danych | Amazon Athena / Azure Synapse |
| **Dataproc** | Managed Spark/Hadoop | Przetwarzanie Big Data na klastrach | Amazon EMR / Azure HDInsight |
| **Cloud Storage (GCS)** | Object storage | Przechowywanie plików (dane, modele, logi) | Amazon S3 / Azure Blob |
| **Dataflow** | Serverless streaming/batch | ETL pipeline'y (Apache Beam) | AWS Glue / Azure Data Factory |

### BigQuery — co to jest?

**BigQuery** to serverless data warehouse od Google. Kluczowe cechy:

- **Serverless** — nie zarządzasz żadną infrastrukturą
- **Niesamowicie szybki** — skanuje terabajty w sekundach dzięki architekturze kolumnowej (Dremel)
- **Pay-per-query** — płacisz za ilość przeskanowanych danych (1 TB = ~$5)
- **Publiczne datasety** — setki bezpłatnych zbiorów danych do eksploracji!
- **Python SDK** — `google-cloud-bigquery` pozwala odpytywać z kodu

### Dataproc — co to jest?

**Dataproc** to zarządzane klastry Apache Spark i Hadoop:

- **Klaster w ~90 sekund** — nie trzeba konfigurować Hadoop ręcznie
- **Płatność za sekundę** — uruchamiasz klaster, robisz obliczenia, wyłączasz
- **Autoscaling** — klaster sam się skaluje w zależności od obciążenia
- **Integracja z GCS** — dane w Cloud Storage, obliczenia na Dataproc

### Kontekst biznesowy

Twoja firma **DataDrive Polska** otrzymała zlecenie od sieci handlowej — analiza danych sprzedażowych z BigQuery public datasets. Klient chce szybkich odpowiedzi na pytania biznesowe, a dane są w chmurze Google. Czas poznać potęgę serverless analytics!

Gotowy/a? Zaczynamy od BigQuery!

---

## Przykład prowadzący — pierwsze zapytanie do BigQuery

Zanim przejdziesz do samodzielnych ćwiczeń, zobaczmy jak wygląda praca z BigQuery z poziomu Pythona.

**Zadanie:** Pobierz 10 najpopularniejszych słów z dzieł Szekspira z publicznego datasetu `bigquery-public-data.samples.shakespeare`.

In [ ]:
# @title Przykład prowadzący — zapytanie do BigQuery

# Krok 1: Tworzymy klienta BigQuery
client = bigquery.Client()

# Krok 2: Piszemy zapytanie SQL
query = """
    SELECT word, SUM(word_count) AS total_count
    FROM `bigquery-public-data.samples.shakespeare`
    GROUP BY word
    ORDER BY total_count DESC
    LIMIT 10
"""

# Krok 3: Wykonujemy zapytanie i pobieramy wyniki do DataFrame
df_shakespeare = client.query(query).to_dataframe()

# Krok 4: Wyświetlamy wyniki
print('Top 10 słów w dziełach Szekspira:')
print(f'   Łączna liczba unikalnych wierszy: {len(df_shakespeare)}')
print()
display(df_shakespeare)

# Krok 5: Szybka wizualizacja
df_shakespeare.plot.barh(x='word', y='total_count', legend=False, color='#4285F4')
plt.title('Top 10 słów w dziełach Szekspira (BigQuery)')
plt.xlabel('Liczba wystąpień')
plt.ylabel('Słowo')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\n✅ Brawo! Właśnie odpytałeś/aś serverless data warehouse w chmurze!')

---

# CZĘŚĆ A — BigQuery z Pythona

W tej części będziesz pracować z **BigQuery** bezpośrednio z Colab. Wszystkie zapytania korzystają z publicznych datasetów Google — nie potrzebujesz płatnego konta!

---

## Ćwiczenie 1: Połączenie z BigQuery (⭐)

Zacznijmy od podstaw — uwierzytelnienie, klient BigQuery i testowe zapytanie.

**Twoje zadania:**

1. Utwórz klienta BigQuery
2. Wylistuj **5 pierwszych datasetów** z projektu `bigquery-public-data` (użyj `client.list_datasets()`)
3. Wykonaj testowe zapytanie do `bigquery-public-data.samples.shakespeare` — policz ile jest **unikalnych dzieł** (kolumna `corpus`)
4. Zapisz wynik w zmiennej `num_works`

**Oczekiwany wynik:**
```
Znalezione datasety: [lista 5 nazw]
Liczba unikalnych dzieł Szekspira: <liczba>
```

In [ ]:
# ✏️ Ćwiczenie 1: Połączenie z BigQuery

# TODO 1: Utwórz klienta BigQuery
client = ...  # Zastąp ... odpowiednim kodem

# TODO 2: Wylistuj 5 pierwszych publicznych datasetów
# Podpowiedź: client.list_datasets('bigquery-public-data') zwraca iterator
datasets = ...  # Zastąp ... — pobierz listę datasetów
dataset_names = ...  # Zastąp ... — wyciągnij nazwy (dataset.dataset_id)

print(f'Znalezione datasety: {dataset_names}')

# TODO 3: Policz unikalne dzieła Szekspira
query_works = """
    -- Zastąp ten komentarz zapytaniem SQL
    -- Użyj COUNT(DISTINCT corpus) z tabeli bigquery-public-data.samples.shakespeare
"""

# TODO 4: Wykonaj zapytanie i zapisz wynik
result_df = ...  # Zastąp ... — wykonaj zapytanie i pobierz DataFrame
num_works = ...  # Zastąp ... — wyciągnij liczbę z DataFrame

print(f'Liczba unikalnych dzieł Szekspira: {num_works}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 1
print("""
Podpowiedź 1: Klient BigQuery tworzy się przez bigquery.Client()

Podpowiedź 2: Iteracja po datasetach:
   datasets_iter = client.list_datasets('bigquery-public-data')
   dataset_names = [ds.dataset_id for ds in datasets_iter]
   Pamiętaj o [:5] żeby wziąć tylko 5 pierwszych!

Podpowiedź 3: Zapytanie SQL:
   SELECT COUNT(DISTINCT corpus) AS num_works
   FROM `bigquery-public-data.samples.shakespeare`

Podpowiedź 4: Wykonanie:
   result_df = client.query(query_works).to_dataframe()
   num_works = result_df['num_works'].iloc[0]
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 1
try:
    assert client is not ..., '❌ Nie utworzyłeś/aś klienta BigQuery'
    assert isinstance(client, bigquery.Client), '❌ client powinien być instancją bigquery.Client'
    assert isinstance(dataset_names, list), '❌ dataset_names powinno być listą'
    assert len(dataset_names) == 5, f'❌ Oczekiwano 5 datasetów, masz {len(dataset_names)}'
    assert num_works is not ..., '❌ Nie policzyłeś/aś dzieł Szekspira'
    assert isinstance(num_works, (int, np.integer)) or num_works > 0, '❌ num_works powinno być liczbą > 0'
    assert num_works == 42, f'❌ Oczekiwano 42 dzieła, masz {num_works}'
    print('✅ Ćwiczenie 1 zaliczone! Połączenie z BigQuery działa!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
    print('Upewnij się, że zdefiniowałeś/aś wszystkie zmienne.')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## Ćwiczenie 2: Eksploracja danych publicznych (⭐)

Czas na prawdziwe dane! Użyjemy datasetu **Chicago Taxi Trips** — jednego z największych publicznych datasetów w BigQuery (~200M wierszy!).

**Tabela:** `bigquery-public-data.chicago_taxi_trips.taxi_trips`

**Twoje zadania:**

1. Policz **łączną liczbę rekordów** (wierszy) w tabeli → `total_rows`
2. Policz **liczbę unikalnych firm taksówkarskich** (kolumna `company`) → `num_companies`
3. Znajdź **zakres dat** — najwcześniejsza i najpóźniejsza data (`trip_start_timestamp`) → `min_date`, `max_date`

⚠️ **WAŻNE:** Używaj `LIMIT` mądrze! Zapytania skanujące całą tabelę mogą zużyć Twój darmowy limit. Funkcje agregujące (COUNT, MIN, MAX) są bezpieczne.

**Oczekiwany wynik:**
```
Łączna liczba wierszy: <duża liczba>
Unikalne firmy taksówkarskie: <liczba>
Zakres dat: <data> — <data>
```

In [ ]:
# Ćwiczenie 2: Eksploracja Chicago Taxi Trips

TABLE = 'bigquery-public-data.chicago_taxi_trips.taxi_trips'

# TODO 1: Policz łączną liczbę wierszy
query_count = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- SELECT COUNT(*) AS total_rows FROM `{TABLE}`
"""
total_rows = ...  # Zastąp ... — wykonaj zapytanie i wyciągnij wynik

# TODO 2: Policz liczbę unikalnych firm
query_companies = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- Użyj COUNT(DISTINCT company)
"""
num_companies = ...  # Zastąp ...

# TODO 3: Znajdź zakres dat
query_dates = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- Użyj MIN() i MAX() na kolumnie trip_start_timestamp
"""
dates_df = ...  # Zastąp ...
min_date = ...  # Zastąp ... — wyciągnij minimalną datę
max_date = ...  # Zastąp ... — wyciągnij maksymalną datę

print(f'Chicago Taxi Trips — eksploracja:')
print(f'   Łączna liczba wierszy: {total_rows:,}')
print(f'   Unikalne firmy taksówkarskie: {num_companies}')
print(f'   Zakres dat: {min_date} — {max_date}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 2
print("""
Podpowiedź 1 (count):
   query_count = f"SELECT COUNT(*) AS total_rows FROM `{TABLE}`"
   df = client.query(query_count).to_dataframe()
   total_rows = df['total_rows'].iloc[0]

Podpowiedź 2 (unikalne firmy):
   SELECT COUNT(DISTINCT company) AS num_companies FROM `{TABLE}`

Podpowiedź 3 (zakres dat):
   SELECT
       MIN(trip_start_timestamp) AS min_date,
       MAX(trip_start_timestamp) AS max_date
   FROM `{TABLE}`
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 2
try:
    assert total_rows is not ..., '❌ Nie policzyłeś/aś wierszy'
    assert isinstance(total_rows, (int, float)), '❌ total_rows powinno być liczbą'
    assert total_rows > 100_000_000, f'❌ Chicago Taxi ma >100M wierszy, masz {total_rows:,}'

    assert num_companies is not ..., '❌ Nie policzyłeś/aś firm'
    assert isinstance(num_companies, (int, float)), '❌ num_companies powinno być liczbą'
    assert num_companies > 10, f'❌ Powinno być więcej niż 10 firm, masz {num_companies}'

    assert min_date is not ..., '❌ Nie znalazłeś/aś minimalnej daty'
    assert max_date is not ..., '❌ Nie znalazłeś/aś maksymalnej daty'

    print(f'✅ Ćwiczenie 2 zaliczone!')
    print(f'   Dane: {total_rows:,} wierszy, {num_companies} firm')
    print(f'   Zakres: {min_date} → {max_date}')
    print('   To jest naprawdę duży dataset!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## Ćwiczenie 3: Analiza SQL w BigQuery (⭐⭐)

Pora na bardziej zaawansowane zapytania! Klient DataDrive Polska chce odpowiedzi na konkretne pytania biznesowe.

**Twoje zadania:**

1. **Top 10 firm taksówkarskich** — znajdź 10 firm z największą liczbą kursów (trip_count) → `df_top_companies`
2. **Średnia opłata wg typu płatności** — oblicz średnią `fare` per `payment_type` → `df_avg_fare`
3. **Najruchliwsze godziny dnia** — pogrupuj kursy wg godziny startu (`EXTRACT(HOUR FROM trip_start_timestamp)`) → `df_hours`

⚠️ Filtruj lata 2022–2023 dla rozsądnego rozmiaru zapytań (`WHERE trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'`)

**Oczekiwany wynik:**
- `df_top_companies` — DataFrame z kolumnami: `company`, `trip_count` (10 wierszy)
- `df_avg_fare` — DataFrame z kolumnami: `payment_type`, `avg_fare`
- `df_hours` — DataFrame z kolumnami: `hour`, `trip_count` (24 wiersze)

In [ ]:
# Ćwiczenie 3: Analiza SQL w BigQuery

TABLE = 'bigquery-public-data.chicago_taxi_trips.taxi_trips'
DATE_FILTER = "trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'"

# TODO 1: Top 10 firm taksówkarskich wg liczby kursów
query_top = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- SELECT company, COUNT(*) AS trip_count
    -- FROM ... WHERE ... AND company IS NOT NULL
    -- GROUP BY ... ORDER BY ... DESC LIMIT 10
"""
df_top_companies = ...  # Zastąp ...

print('Top 10 firm taksówkarskich (2022-2023):')
display(df_top_companies)

# TODO 2: Średnia opłata wg typu płatności
query_fare = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- SELECT payment_type, ROUND(AVG(fare), 2) AS avg_fare
    -- FROM ... WHERE ... AND payment_type IS NOT NULL AND fare > 0
    -- GROUP BY ... ORDER BY avg_fare DESC
"""
df_avg_fare = ...  # Zastąp ...

print('\nŚrednia opłata wg typu płatności:')
display(df_avg_fare)

# TODO 3: Najruchliwsze godziny dnia
query_hours = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- SELECT EXTRACT(HOUR FROM trip_start_timestamp) AS hour,
    --        COUNT(*) AS trip_count
    -- FROM ... WHERE ...
    -- GROUP BY hour ORDER BY hour
"""
df_hours = ...  # Zastąp ...

print('\nKursy wg godziny dnia:')
display(df_hours)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 3
print("""
Podpowiedź 1 (Top 10 firm):
   SELECT company, COUNT(*) AS trip_count
   FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
   WHERE trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'
     AND company IS NOT NULL
   GROUP BY company
   ORDER BY trip_count DESC
   LIMIT 10

Podpowiedź 2 (Średnia fare):
   SELECT payment_type, ROUND(AVG(fare), 2) AS avg_fare
   FROM ... WHERE ... AND payment_type IS NOT NULL AND fare > 0
   GROUP BY payment_type
   ORDER BY avg_fare DESC

Podpowiedź 3 (Godziny):
   SELECT EXTRACT(HOUR FROM trip_start_timestamp) AS hour,
          COUNT(*) AS trip_count
   FROM ... WHERE ...
   GROUP BY hour
   ORDER BY hour

Pamiętaj: df = client.query(query).to_dataframe()
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 3
try:
    # Test top companies
    assert df_top_companies is not ..., '❌ Nie wykonałeś/aś zapytania o top firmy'
    assert isinstance(df_top_companies, pd.DataFrame), '❌ df_top_companies powinien być DataFrame'
    assert len(df_top_companies) == 10, f'❌ Oczekiwano 10 firm, masz {len(df_top_companies)}'
    assert 'company' in df_top_companies.columns, '❌ Brakuje kolumny company'
    assert 'trip_count' in df_top_companies.columns, '❌ Brakuje kolumny trip_count'
    print('✅ Top 10 firm — OK!')

    # Test avg fare
    assert df_avg_fare is not ..., '❌ Nie wykonałeś/aś zapytania o avg fare'
    assert isinstance(df_avg_fare, pd.DataFrame), '❌ df_avg_fare powinien być DataFrame'
    assert 'payment_type' in df_avg_fare.columns, '❌ Brakuje kolumny payment_type'
    assert 'avg_fare' in df_avg_fare.columns, '❌ Brakuje kolumny avg_fare'
    print('✅ Średnia fare wg płatności — OK!')

    # Test hours
    assert df_hours is not ..., '❌ Nie wykonałeś/aś zapytania o godziny'
    assert isinstance(df_hours, pd.DataFrame), '❌ df_hours powinien być DataFrame'
    assert 'hour' in df_hours.columns, '❌ Brakuje kolumny hour'
    assert 'trip_count' in df_hours.columns, '❌ Brakuje kolumny trip_count'
    assert len(df_hours) == 24, f'❌ Powinno być 24 godziny, masz {len(df_hours)}'
    print('✅ Godziny szczytu — OK!')

    print('\nĆwiczenie 3 zaliczone! SQL w BigQuery opanowany!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## Ćwiczenie 4: Zaawansowane zapytania i wizualizacja (⭐⭐)

Klient chce zobaczyć **trendy** i **rozkłady** — pora na wizualizację danych z chmury!

**Twoje zadania:**

1. **Trend miesięczny** — policz liczbę kursów per miesiąc (2022–2023), zapisz w `df_monthly`
   - Kolumny: `month` (format `YYYY-MM`), `trip_count`
   - Narysuj wykres liniowy trendu

2. **Rozkład opłat** — pobierz 10 000 losowych wartości `fare` (z zakresu 1–100 USD, lata 2022–2023) do `df_fares`
   - Narysuj histogram rozkładu opłat
   - Oblicz i wypisz: średnią, medianę, odchylenie standardowe

**Oczekiwany wynik:** Dwa wykresy (liniowy + histogram) oraz statystyki opisowe.

In [ ]:
# Ćwiczenie 4: Zaawansowane zapytania i wizualizacja

TABLE = 'bigquery-public-data.chicago_taxi_trips.taxi_trips'

# === CZĘŚĆ 1: Trend miesięczny ===

# TODO 1: Zapytanie — liczba kursów per miesiąc
query_monthly = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- Użyj FORMAT_TIMESTAMP('%Y-%m', trip_start_timestamp) AS month
    -- COUNT(*) AS trip_count
    -- Filtruj lata 2022-2023, GROUP BY month, ORDER BY month
"""
df_monthly = ...  # Zastąp ...

# TODO 2: Wykres liniowy trendu
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Lewy wykres — trend miesięczny
# Użyj axes[0].plot() lub df_monthly.plot() z ax=axes[0]
# ...
axes[0].set_title('Trend miesięczny — Chicago Taxi (2022-2023)')
axes[0].set_xlabel('Miesiąc')
axes[0].set_ylabel('Liczba kursów')
axes[0].tick_params(axis='x', rotation=45)

# === CZĘŚĆ 2: Rozkład opłat ===

# TODO 3: Pobierz 10 000 losowych wartości fare
query_fares = f"""
    -- Zastąp ten komentarz zapytaniem SQL
    -- SELECT fare FROM ... WHERE fare BETWEEN 1 AND 100
    -- AND trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'
    -- ORDER BY RAND() LIMIT 10000
"""
df_fares = ...  # Zastąp ...

# TODO 4: Histogram rozkładu opłat (axes[1])
# Użyj axes[1].hist() lub df_fares['fare'].hist(ax=axes[1])
# ...
axes[1].set_title('Rozkład opłat za kurs (próbka 10K)')
axes[1].set_xlabel('Opłata (USD)')
axes[1].set_ylabel('Liczba kursów')

plt.tight_layout()
plt.show()

# TODO 5: Statystyki opisowe
mean_fare = ...  # Zastąp ... — średnia opłata
median_fare = ...  # Zastąp ... — mediana opłaty
std_fare = ...  # Zastąp ... — odchylenie standardowe

print(f'\nStatystyki opłat (próbka 10K kursów):')
print(f'   Średnia:    ${mean_fare:.2f}')
print(f'   Mediana:    ${median_fare:.2f}')
print(f'   Std dev:    ${std_fare:.2f}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 4
print("""
Podpowiedź 1 (trend miesięczny):
   SELECT FORMAT_TIMESTAMP('%Y-%m', trip_start_timestamp) AS month,
          COUNT(*) AS trip_count
   FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
   WHERE trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'
   GROUP BY month
   ORDER BY month

Podpowiedź 2 (wykres liniowy):
   axes[0].plot(df_monthly['month'], df_monthly['trip_count'],
                marker='o', color='#4285F4')

Podpowiedź 3 (losowe fare):
   SELECT fare
   FROM `...`
   WHERE fare BETWEEN 1 AND 100
     AND trip_start_timestamp BETWEEN '2022-01-01' AND '2023-12-31'
   ORDER BY RAND()
   LIMIT 10000

Podpowiedź 4 (histogram):
   df_fares['fare'].hist(ax=axes[1], bins=50, color='#34A853', edgecolor='white')

Podpowiedź 5 (statystyki):
   mean_fare = df_fares['fare'].mean()
   median_fare = df_fares['fare'].median()
   std_fare = df_fares['fare'].std()
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 4
try:
    # Test monthly
    assert df_monthly is not ..., '❌ Nie wykonałeś/aś zapytania o trend miesięczny'
    assert isinstance(df_monthly, pd.DataFrame), '❌ df_monthly powinien być DataFrame'
    assert 'month' in df_monthly.columns, '❌ Brakuje kolumny month'
    assert 'trip_count' in df_monthly.columns, '❌ Brakuje kolumny trip_count'
    assert len(df_monthly) >= 20, f'❌ Oczekiwano ~24 miesiące, masz {len(df_monthly)}'
    print('✅ Trend miesięczny — OK!')

    # Test fares
    assert df_fares is not ..., '❌ Nie pobrałeś/aś opłat'
    assert isinstance(df_fares, pd.DataFrame), '❌ df_fares powinien być DataFrame'
    assert len(df_fares) == 10_000, f'❌ Oczekiwano 10000 wierszy, masz {len(df_fares)}'
    print('✅ Próbka opłat — OK!')

    # Test stats
    assert mean_fare is not ..., '❌ Nie policzyłeś/aś średniej'
    assert median_fare is not ..., '❌ Nie policzyłeś/aś mediany'
    assert std_fare is not ..., '❌ Nie policzyłeś/aś odchylenia'
    assert 5 < mean_fare < 50, f'❌ Średnia opłata wygląda dziwnie: ${mean_fare:.2f}'
    print('✅ Statystyki — OK!')

    print('\nĆwiczenie 4 zaliczone! Piękne wykresy z danych chmurowych!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

# CZĘŚĆ B — Dataproc i Spark w chmurze

W tej części przejdziemy przez proces pracy z **Google Cloud Dataproc** — zarządzanymi klastrami Spark/Hadoop.

> ⚠️ **Uwaga:** Część B jest typu **walkthrough** — komendy `gcloud` prezentujemy jako kod do przeanalizowania. Jeśli masz aktywne konto GCP z billingiem, możesz je uruchomić bezpośrednio. W przeciwnym razie masz **3 darmowe opcje** poniżej!

---

## 🆓 Jak uruchomić te komendy BEZ karty kredytowej?

### ⭐ Opcja 1: Cloud Skills Boost (NAJLEPSZA — polecana!)

Platforma **[cloudskillsboost.google](https://www.cloudskillsboost.google/)** daje Ci **tymczasowy projekt GCP z aktywnym billingiem** — bez Twojej karty kredytowej!

**Idealne laby do tego ćwiczenia:**

| Lab (darmowy) | Co ćwiczysz | Link |
|---------------|-------------|------|
| **Dataproc: Qwik Start - Command Line** | `gcloud dataproc clusters create`, `gcloud dataproc jobs submit` | [→ cloudskillsboost](https://www.cloudskillsboost.google/focuses/585) |
| **Dataproc: Qwik Start - Console** | Tworzenie klastra przez GUI | [→ cloudskillsboost](https://www.cloudskillsboost.google/focuses/586) |
| **Running Apache Spark jobs on Cloud Dataproc** | Pełny flow: klaster → job → wyniki | [→ cloudskillsboost](https://www.cloudskillsboost.google/focuses/672) |

### 🖥️ Opcja 2: Google Cloud Shell (terminal w przeglądarce)

[shell.cloud.google.com](https://shell.cloud.google.com) — **darmowy terminal z gcloud CLI** dla każdego użytkownika Google (wystarczy konto Gmail)!

```bash
# Dostępne natychmiast — bez instalacji, bez karty:
gcloud --version           # ✅ gcloud CLI gotowy
bq ls                      # ✅ BigQuery CLI
python3 -c "import pyspark"  # ❌ trzeba instalować
pip install pyspark          # ale można!
```

> ⚠️ **Ograniczenie:** Cloud Shell nie ma własnego projektu z billingiem — komendy `gcloud dataproc` nie zadziałają bez projektu. Użyj Cloud Skills Boost do pełnego Dataproc!

### 📊 Opcja 3: BigQuery Sandbox (tylko Część A)

BigQuery działa bez billing account w trybie sandbox:
- ✅ 1 TB zapytań miesięcznie za darmo
- ✅ Dostęp do publicznych datasetów
- Otwórz: [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)

---

### Architektura Dataproc

```
┌──────────────────────────────────────────────────────┐
│                  Google Cloud Platform                │
│                                                      │
│  ┌──────────────────┐    ┌──────────────────────┐   │
│  │  Cloud Storage   │    │   Dataproc Cluster    │   │
│  │  (GCS)           │◄──►│                       │   │
│  │  gs://bucket/    │    │  Master (1x)          │   │
│  │  ├── input/      │    │  Workers (2-100x)     │   │
│  │  ├── output/     │    │  [Spark, Hadoop,      │   │
│  │  └── scripts/    │    │   Hive, Pig, ...]     │   │
│  └──────────────────┘    └──────────────────────┘   │
│                                                      │
│  ┌──────────────────┐    ┌──────────────────────┐   │
│  │  BigQuery        │    │   Cloud IAM           │   │
│  │  (results)       │    │   (uprawnienia)       │   │
│  └──────────────────┘    └──────────────────────┘   │
└──────────────────────────────────────────────────────┘
```

**Flow pracy z Dataproc:**
1. Wgraj dane i skrypty do **Cloud Storage** (GCS)
2. Utwórz **klaster Dataproc** (master + workers)
3. Wyślij **job PySpark** do klastra
4. Pobierz wyniki z GCS lub BigQuery
5. **Usuń klaster** (płacisz za minuty!)

---

## Ćwiczenie 5: Dataproc — tworzenie klastra (walkthrough) (⭐⭐)

Przeanalizujmy krok po kroku, jak stworzyć klaster Dataproc z poziomu wiersza poleceń `gcloud`.

**Twoje zadania (analiza kodu):**

1. Przeczytaj komendy poniżej i **zrozum każdą flagę**
2. Wypełnij **brakujące wartości** w komórkach z TODO
3. Odpowiedz na pytania kontrolne

In [ ]:
# Ćwiczenie 5: Komendy gcloud do tworzenia klastra Dataproc
# ⚠️ NIE URUCHAMIAJ tych komend — analizuj je i wypełnij luki!

# --- Krok 1: Ustawienie projektu GCP ---
cmd_set_project = """
gcloud config set project MY_PROJECT_ID
"""

# --- Krok 2: Utworzenie bucketu GCS na dane ---
cmd_create_bucket = """
gcloud storage buckets create gs://datadrive-polska-bucket \\
    --location=europe-central2 \\
    --uniform-bucket-level-access
"""

# --- Krok 3: Tworzenie klastra Dataproc ---
cmd_create_cluster = """
gcloud dataproc clusters create datadrive-cluster \\
    --region=europe-central2 \\
    --zone=europe-central2-a \\
    --master-machine-type=n1-standard-4 \\
    --master-boot-disk-size=100GB \\
    --num-workers=2 \\
    --worker-machine-type=n1-standard-4 \\
    --worker-boot-disk-size=100GB \\
    --image-version=2.1-debian11 \\
    --optional-components=JUPYTER \\
    --enable-component-gateway \\
    --max-idle=30m
"""

# --- Krok 4: Sprawdzenie statusu klastra ---
cmd_describe_cluster = """
gcloud dataproc clusters describe datadrive-cluster \\
    --region=europe-central2
"""

print('Komendy gcloud do analizy:')
print('=' * 50)
print('1. Set project:', cmd_set_project.strip())
print('2. Create bucket:', cmd_create_bucket.strip()[:60], '...')
print('3. Create cluster:', cmd_create_cluster.strip()[:60], '...')
print('4. Describe cluster:', cmd_describe_cluster.strip()[:60], '...')
print('\n✅ Przeanalizuj komendy powyżej i odpowiedz na pytania poniżej!')

In [ ]:
# ✏️ Ćwiczenie 5: Pytania kontrolne — wypełnij odpowiedzi

# TODO: Wypełnij odpowiedzi na pytania (wpisz stringi)

# Pytanie 1: Ile węzłów worker ma klaster?
answer_num_workers = ...  # Zastąp ... liczbą

# Pytanie 2: Jaki typ maszyny ma master node?
answer_master_type = ...  # Zastąp ... stringiem, np. 'n1-standard-2'

# Pytanie 3: W jakim regionie GCP tworzony jest klaster?
answer_region = ...  # Zastąp ... stringiem

# Pytanie 4: Po jakim czasie bezczynności klaster się wyłączy (auto-delete)?
answer_idle_time = ...  # Zastąp ... stringiem, np. '10m'

# Pytanie 5: Jaki opcjonalny komponent jest zainstalowany na klastrze?
answer_component = ...  # Zastąp ... stringiem

# Pytanie 6: Dlaczego używamy flagi --max-idle?
# Odpowiedz w 1-2 zdaniach po polsku
answer_why_idle = ...  # Zastąp ... stringiem z wyjaśnieniem

print('Twoje odpowiedzi:')
print(f'  1. Workers: {answer_num_workers}')
print(f'  2. Master type: {answer_master_type}')
print(f'  3. Region: {answer_region}')
print(f'  4. Idle time: {answer_idle_time}')
print(f'  5. Component: {answer_component}')
print(f'  6. Dlaczego idle: {answer_why_idle}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 5
print("""
Podpowiedź: Przeczytaj flagi w komendzie gcloud dataproc clusters create:

   --num-workers=???       → ile workerów?
   --master-machine-type=??? → jaki typ maszyny master?
   --region=???            → jaki region?
   --max-idle=???          → po jakim czasie idle się wyłączy?
   --optional-components=??? → jaki komponent?

Flaga --max-idle jest ważna, bo klastry Dataproc kosztują pieniądze.
Jeśli nikt nie korzysta, lepiej je wyłączyć automatycznie!
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 5
try:
    assert answer_num_workers is not ..., '❌ Nie podałeś/aś liczby workerów'
    assert answer_num_workers == 2, f'❌ Liczba workerów to 2, nie {answer_num_workers}'
    print('✅ Liczba workerów — OK!')

    assert answer_master_type is not ..., '❌ Nie podałeś/aś typu maszyny master'
    assert answer_master_type == 'n1-standard-4', f'❌ Sprawdź flagę --master-machine-type'
    print('✅ Typ maszyny master — OK!')

    assert answer_region is not ..., '❌ Nie podałeś/aś regionu'
    assert answer_region == 'europe-central2', f'❌ Region to europe-central2 (Warszawa!)'
    print('✅ Region — OK! (Warszawa)')

    assert answer_idle_time is not ..., '❌ Nie podałeś/aś czasu idle'
    assert answer_idle_time == '30m', f'❌ Czas idle to 30m, nie {answer_idle_time}'
    print('✅ Idle time — OK!')

    assert answer_component is not ..., '❌ Nie podałeś/aś komponentu'
    assert answer_component.upper() == 'JUPYTER', f'❌ Komponent to JUPYTER, nie {answer_component}'
    print('✅ Komponent — OK!')

    assert answer_why_idle is not ..., '❌ Nie wyjaśniłeś/aś dlaczego --max-idle'
    assert isinstance(answer_why_idle, str) and len(answer_why_idle) > 10, '❌ Odpowiedź za krótka'
    print('✅ Wyjaśnienie — OK!')

    print('\nĆwiczenie 5 zaliczone! Wiesz jak stworzyć klaster Dataproc!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## Ćwiczenie 6: Submittowanie PySpark job na Dataproc (⭐⭐⭐)

Teraz napiszemy kompletny **skrypt PySpark**, który mógłby być wysłany do klastra Dataproc. Następnie przeanalizujemy komendę `gcloud` do submittowania joba.

**Scenariusz:** DataDrive Polska otrzymała plik CSV z danymi sprzedażowymi na Cloud Storage (`gs://datadrive-polska-bucket/input/sales.csv`). Trzeba go przetworzyć Sparkiem na klastrze Dataproc.

**Twoje zadania:**

1. Uzupełnij skrypt PySpark przetwarzający dane sprzedaży
2. Przeanalizuj komendę `gcloud` do wysłania joba
3. Odpowiedz na pytania kontrolne

In [ ]:
# Ćwiczenie 6: Kompletny skrypt PySpark dla Dataproc
# Ten skrypt zapisalibyśmy jako plik .py i wgrali do GCS

pyspark_job_script = '''
#!/usr/bin/env python3
"""sales_analysis.py — Analiza sprzedaży na Dataproc.

Uruchamiany jako job PySpark na klastrze Dataproc.
Czyta dane z GCS, przetwarza i zapisuje wyniki z powrotem do GCS.
"""
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import sys

def main(input_path: str, output_path: str) -> None:
    """Główna funkcja przetwarzania danych sprzedażowych."""

    # Krok 1: Inicjalizacja SparkSession
    spark = (
        SparkSession.builder
        .appName("DataDrive-SalesAnalysis")
        .getOrCreate()
    )

    print(f"Czytam dane z: {input_path}")

    # Krok 2: Wczytanie danych z GCS
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(input_path)
    )

    print(f"   Wierszy: {df.count()}, Kolumn: {len(df.columns)}")

    # Krok 3: Agregacja — przychód per kategoria produktu
    revenue_by_category = (
        df
        .groupBy("product_category")
        .agg(
            F.sum("amount").alias("total_revenue"),
            F.count("*").alias("num_transactions"),
            F.avg("amount").alias("avg_transaction"),
        )
        .orderBy(F.desc("total_revenue"))
    )

    # Krok 4: Top 5 miast wg sprzedaży
    top_cities = (
        df
        .groupBy("city")
        .agg(F.sum("amount").alias("total_revenue"))
        .orderBy(F.desc("total_revenue"))
        .limit(5)
    )

    # Krok 5: Zapis wyników do GCS (format Parquet)
    output_revenue = f"{output_path}/revenue_by_category"
    output_cities = f"{output_path}/top_cities"

    revenue_by_category.write.mode("overwrite").parquet(output_revenue)
    top_cities.write.mode("overwrite").parquet(output_cities)

    print(f"✅ Wyniki zapisane do: {output_path}")
    print(f"   - {output_revenue}")
    print(f"   - {output_cities}")

    spark.stop()

if __name__ == "__main__":
    input_path = sys.argv[1]   # gs://bucket/input/sales.csv
    output_path = sys.argv[2]  # gs://bucket/output/
    main(input_path, output_path)
'''

print('Skrypt PySpark (sales_analysis.py):')
print('=' * 50)
print(pyspark_job_script)
print('=' * 50)

In [ ]:
# Ćwiczenie 6 (cd.): Komenda gcloud do wysłania joba

# Krok 1: Wgranie skryptu do GCS
cmd_upload_script = """
gcloud storage cp sales_analysis.py gs://datadrive-polska-bucket/scripts/
"""

# Krok 2: Submittowanie joba PySpark
cmd_submit_job = """
gcloud dataproc jobs submit pyspark \\
    gs://datadrive-polska-bucket/scripts/sales_analysis.py \\
    --cluster=datadrive-cluster \\
    --region=europe-central2 \\
    -- gs://datadrive-polska-bucket/input/sales.csv \\
       gs://datadrive-polska-bucket/output/
"""

# Krok 3: Sprawdzenie statusu joba
cmd_check_job = """
gcloud dataproc jobs list \\
    --cluster=datadrive-cluster \\
    --region=europe-central2 \\
    --state-filter=active
"""

# Krok 4: Pobranie wyników
cmd_get_results = """
gcloud storage ls gs://datadrive-polska-bucket/output/
gcloud storage cp -r gs://datadrive-polska-bucket/output/ ./local_results/
"""

# Krok 5: Usunięcie klastra (WAŻNE — żeby nie płacić!)
cmd_delete_cluster = """
gcloud dataproc clusters delete datadrive-cluster \\
    --region=europe-central2 \\
    --quiet
"""

print('Pipeline Dataproc — komendy gcloud:')
steps = [
    ('Upload skryptu do GCS', cmd_upload_script),
    ('Submit PySpark job', cmd_submit_job),
    ('Sprawdzenie statusu', cmd_check_job),
    ('Pobranie wyników', cmd_get_results),
    ('Usunięcie klastra ⚠️', cmd_delete_cluster),
]
for i, (name, cmd) in enumerate(steps, 1):
    print(f'\n--- Krok {i}: {name} ---')
    print(cmd.strip())

In [ ]:
# ✏️ Ćwiczenie 6 (cd.): Pytania kontrolne

# TODO: Wypełnij odpowiedzi

# Pytanie 1: W jakim formacie skrypt zapisuje wyniki? (np. 'csv', 'parquet', 'json')
answer_output_format = ...  # Zastąp ...

# Pytanie 2: Jaka komenda gcloud służy do wysyłania joba PySpark?
# (wpisz pełną komendę bazową, np. 'gcloud dataproc ...')
answer_submit_cmd = ...  # Zastąp ...

# Pytanie 3: Dlaczego ZAWSZE powinniśmy usuwać klaster po zakończeniu pracy?
answer_why_delete = ...  # Zastąp ... stringiem z wyjaśnieniem

# Pytanie 4: Co robi separator '--' (podwójny myślnik) w komendzie submit?
# (oddziela flagi gcloud od argumentów skryptu Python)
answer_separator = ...  # Zastąp ... stringiem z wyjaśnieniem

print('Twoje odpowiedzi:')
print(f'  1. Format wyjścia: {answer_output_format}')
print(f'  2. Komenda submit: {answer_submit_cmd}')
print(f'  3. Dlaczego usuwać: {answer_why_delete}')
print(f'  4. Separator --: {answer_separator}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 6
print("""
Podpowiedź 1: Sprawdź linię z .write.mode("overwrite")... — jaki format?

Podpowiedź 2: Komenda zaczyna się od 'gcloud dataproc jobs submit pyspark'

Podpowiedź 3: Klastry Dataproc kosztują $$$ za każdą minutę działania.
   Nawet idle klaster zużywa Compute Engine VMs!

Podpowiedź 4: W komendzie `gcloud ... -- arg1 arg2`
   separator -- oddziela flagi gcloud od parametrów
   przekazywanych do skryptu Pythona (sys.argv).
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 6
try:
    assert answer_output_format is not ..., '❌ Nie podałeś/aś formatu wyjścia'
    assert answer_output_format.lower() == 'parquet', f'❌ Format to parquet, nie {answer_output_format}'
    print('✅ Format wyjścia — OK!')

    assert answer_submit_cmd is not ..., '❌ Nie podałeś/aś komendy submit'
    assert 'gcloud dataproc jobs submit pyspark' in answer_submit_cmd.lower(), \
        '❌ Komenda powinna zawierać: gcloud dataproc jobs submit pyspark'
    print('✅ Komenda submit — OK!')

    assert answer_why_delete is not ..., '❌ Nie wyjaśniłeś/aś dlaczego usuwać klaster'
    assert isinstance(answer_why_delete, str) and len(answer_why_delete) > 10, \
        '❌ Odpowiedź za krótka'
    print('✅ Wyjaśnienie usuwania — OK!')

    assert answer_separator is not ..., '❌ Nie wyjaśniłeś/aś separatora --'
    assert isinstance(answer_separator, str) and len(answer_separator) > 10, \
        '❌ Odpowiedź za krótka'
    print('✅ Separator -- — OK!')

    print('\nĆwiczenie 6 zaliczone! Wiesz jak submitować joby na Dataproc!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## Ćwiczenie 7: Optymalizacja kosztów chmurowych (⭐⭐⭐)

Szef DataDrive Polska pyta: **"Ile to będzie kosztować?"**

To pytanie, które ZAWSZE musisz umieć odpowiedzieć w chmurze! Policzmy koszty różnych scenariuszy.

### Cennik GCP (uproszczony, ceny przybliżone)

| Usługa | Model cenowy | Cena |
|--------|-------------|------|
| **BigQuery** — On-demand | Per TB przeskanowanych danych | $5.00 / TB |
| **BigQuery** — Flat-rate (editions) | Sloty obliczeniowe per godzinę | $0.04 / slot-hour (Standard) |
| **Dataproc** — Standard worker | Per vCPU per godzina (oprócz ceny VM) | $0.010 / vCPU / godz. |
| **Dataproc** — Preemptible worker | Tańsze, ale mogą zostać wyłączone | $0.003 / vCPU / godz. |
| **Compute Engine** — n1-standard-4 | 4 vCPU, ~15 GB RAM | ~$0.19 / godz. |
| **Cloud Storage** | Per GB przechowywania + operacje | $0.020 / GB / miesiąc |

**Twoje zadanie:** Wypełnij wzory kosztów i policz scenariusze.

In [ ]:
# Ćwiczenie 7: Kalkulator kosztów chmurowych

# === SCENARIUSZ A: BigQuery ===

# Ceny BigQuery
BQ_PRICE_PER_TB = 5.00  # USD per TB (on-demand)

# Firma skanuje dziennie:
daily_scan_gb = 500  # GB dziennie
working_days_per_month = 22

# TODO 1: Oblicz miesięczny koszt BigQuery (on-demand)
monthly_scan_tb = ...  # Zastąp ... — przelicz GB na TB
monthly_bq_cost = ...  # Zastąp ... — koszt = TB * cena_per_TB

print('=== SCENARIUSZ A: BigQuery On-Demand ===')
print(f'  Dzienne skanowanie: {daily_scan_gb} GB')
print(f'  Miesięczne skanowanie: {monthly_scan_tb:.1f} TB')
print(f'  Miesięczny koszt: ${monthly_bq_cost:.2f}')

# === SCENARIUSZ B: Dataproc — standard vs preemptible ===

# Konfiguracja klastra
num_workers = 4
vcpus_per_worker = 4  # n1-standard-4 ma 4 vCPU
hours_per_day = 8
days_per_month = 22

# Ceny
COMPUTE_ENGINE_PRICE = 0.19  # per godz. per VM (n1-standard-4)
DATAPROC_STANDARD_PRICE = 0.010  # per vCPU per godz.
DATAPROC_PREEMPTIBLE_PRICE = 0.003  # per vCPU per godz.

total_hours = hours_per_day * days_per_month
total_vcpus = num_workers * vcpus_per_worker

# TODO 2: Koszt samego Compute Engine (VMs dla workerów + 1 master)
num_vms = ...  # Zastąp ... — workers + 1 master
compute_cost = ...  # Zastąp ... — num_vms * price * total_hours

# TODO 3: Dodatkowy koszt Dataproc (standard workers)
dataproc_standard_cost = ...  # Zastąp ... — total_vcpus * standard_price * total_hours

# TODO 4: Dodatkowy koszt Dataproc (preemptible workers)
dataproc_preemptible_cost = ...  # Zastąp ... — total_vcpus * preemptible_price * total_hours

# TODO 5: Łączne koszty obu wariantów
total_standard = ...  # Zastąp ... — compute + dataproc_standard
total_preemptible = ...  # Zastąp ... — compute + dataproc_preemptible
savings = ...  # Zastąp ... — ile oszczędzamy z preemptible?

print(f'\n=== SCENARIUSZ B: Dataproc ({num_workers} workers, {hours_per_day}h/dzień) ===')
print(f'  Compute Engine (VMs): ${compute_cost:.2f}/mies.')
print(f'  Dataproc (standard):  ${dataproc_standard_cost:.2f}/mies.')
print(f'  Dataproc (preempt.):  ${dataproc_preemptible_cost:.2f}/mies.')
print(f'  ---')
print(f'  Total (standard):     ${total_standard:.2f}/mies.')
print(f'  Total (preemptible):  ${total_preemptible:.2f}/mies.')
print(f'  Oszczędność:          ${savings:.2f}/mies. ({savings/total_standard*100:.1f}%)')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 7
print("""
Podpowiedź 1 (BigQuery):
   monthly_scan_tb = (daily_scan_gb * working_days_per_month) / 1000
   monthly_bq_cost = monthly_scan_tb * BQ_PRICE_PER_TB

Podpowiedź 2 (Compute Engine):
   num_vms = num_workers + 1  (workers + master)
   compute_cost = num_vms * COMPUTE_ENGINE_PRICE * total_hours

Podpowiedź 3 (Dataproc standard):
   dataproc_standard_cost = total_vcpus * DATAPROC_STANDARD_PRICE * total_hours

Podpowiedź 4 (Dataproc preemptible):
   dataproc_preemptible_cost = total_vcpus * DATAPROC_PREEMPTIBLE_PRICE * total_hours

Podpowiedź 5 (Totals):
   total_standard = compute_cost + dataproc_standard_cost
   total_preemptible = compute_cost + dataproc_preemptible_cost
   savings = total_standard - total_preemptible
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 7
try:
    # BigQuery
    expected_scan_tb = (500 * 22) / 1000
    expected_bq_cost = expected_scan_tb * 5.00

    assert monthly_scan_tb is not ..., '❌ Nie obliczyłeś/aś monthly_scan_tb'
    assert abs(monthly_scan_tb - expected_scan_tb) < 0.01, \
        f'❌ monthly_scan_tb powinno być {expected_scan_tb}, masz {monthly_scan_tb}'
    assert abs(monthly_bq_cost - expected_bq_cost) < 0.01, \
        f'❌ monthly_bq_cost powinno być ${expected_bq_cost}, masz ${monthly_bq_cost}'
    print('✅ BigQuery on-demand — OK!')

    # Compute
    expected_vms = 5
    expected_compute = expected_vms * 0.19 * (8 * 22)
    assert num_vms == expected_vms, f'❌ num_vms powinno być {expected_vms}'
    assert abs(compute_cost - expected_compute) < 0.01, \
        f'❌ compute_cost powinno być ${expected_compute:.2f}'
    print('✅ Compute Engine — OK!')

    # Dataproc
    expected_dp_std = 16 * 0.010 * 176
    expected_dp_pre = 16 * 0.003 * 176
    assert abs(dataproc_standard_cost - expected_dp_std) < 0.01, \
        f'❌ dataproc_standard_cost powinno być ${expected_dp_std:.2f}'
    assert abs(dataproc_preemptible_cost - expected_dp_pre) < 0.01, \
        f'❌ dataproc_preemptible_cost powinno być ${expected_dp_pre:.2f}'
    print('✅ Dataproc (standard vs preemptible) — OK!')

    # Totals
    expected_total_std = expected_compute + expected_dp_std
    expected_total_pre = expected_compute + expected_dp_pre
    expected_savings = expected_total_std - expected_total_pre
    assert abs(total_standard - expected_total_std) < 0.01, '❌ total_standard niepoprawne'
    assert abs(total_preemptible - expected_total_pre) < 0.01, '❌ total_preemptible niepoprawne'
    assert abs(savings - expected_savings) < 0.01, '❌ savings niepoprawne'
    print('✅ Łączne koszty i oszczędności — OK!')

    print(f'\nĆwiczenie 7 zaliczone! Potrafisz liczyć koszty chmury!')
    print(f'   BigQuery: ${monthly_bq_cost:.2f}/mies.')
    print(f'   Dataproc standard: ${total_standard:.2f}/mies.')
    print(f'   Dataproc preemptible: ${total_preemptible:.2f}/mies.')
    print(f'   Oszczędność z preemptible: ${savings:.2f} ({savings/total_standard*100:.1f}%)')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie!')
except NameError as e:
    print(f'❌ Brakująca zmienna: {e}')
except Exception as e:
    print(f'❌ Błąd: {e}')

---

## 🎓 Bonus: Cloud Skills Boost — Prawdziwa chmura bez karty kredytowej!

Google oferuje **darmowe laby** w [Cloud Skills Boost](https://www.cloudskillsboost.google/) (dawniej QwikLabs) — dostajesz tymczasowe konto GCP z billingiem. Bez Twojej karty, bez ryzyka kosztów!

### 🟢 Darmowe laby (zacznij od tych!):

| Lab | Czas | Co ćwiczysz |
|-----|------|-------------|
| [Dataproc: Qwik Start - Command Line](https://www.cloudskillsboost.google/focuses/585) | ~30 min | `gcloud dataproc` — DOKŁADNIE komendy z tego ćwiczenia! |
| [Dataproc: Qwik Start - Console](https://www.cloudskillsboost.google/focuses/586) | ~30 min | Klaster przez GUI konsoli GCP |
| [BigQuery: Qwik Start - Console](https://www.cloudskillsboost.google/focuses/1145) | ~30 min | SQL na prawdziwych publicznych danych |
| [Cloud Storage: Qwik Start - Console](https://www.cloudskillsboost.google/focuses/1836) | ~30 min | Bucket, upload, uprawnienia |

### 🔵 Zaawansowane laby:

| Lab/Quest | Czas | Poziom |
|-----------|------|--------|
| [Running Apache Spark jobs on Cloud Dataproc](https://www.cloudskillsboost.google/focuses/672) | ~1h | Intermediate |
| [Perform Foundational Data, ML, and AI Tasks in Google Cloud](https://www.cloudskillsboost.google/course_templates/631) | ~6h | Intermediate |
| [Data Engineering on Google Cloud](https://www.cloudskillsboost.google/paths/16) | ~16h | Advanced |

### Jak zacząć?

1. Wejdź na [cloudskillsboost.google](https://www.cloudskillsboost.google/)
2. Zaloguj się kontem Google
3. Wybierz darmowy lab (🟢 = Free)
4. Kliknij "Start Lab" → dostaniesz tymczasowe konto GCP!
5. Po ukończeniu — badge na LinkedIn 🎖️

### Zadanie bonusowe

Ukończ lab **"Dataproc: Qwik Start - Command Line"** i wklej link do swojego profilu lub badge'a poniżej.

> 💡 **Protip:** Cloud Skills Boost daje Ci **prawdziwe środowisko GCP** z aktywnym billingiem — zupełnie za darmo! Komendy z tego ćwiczenia (Część B) zadziałają tam 1:1. To najlepszy sposób na praktykę bez ryzyka kosztów!

In [ ]:
# Bonus: Wklej tutaj link do badge'a lub screenshot

# Jeśli ukończyłeś/aś lab w Cloud Skills Boost,
# wklej link do swojego publicznego profilu lub badge'a:
badge_link = ''  # Wklej swój link tutaj

# Lub wgraj screenshot:
# from google.colab import files
# uploaded = files.upload()  # Odkomentuj żeby wgrać plik

if badge_link:
    print(f'Twój badge: {badge_link}')
    print('Brawo! Ukończyłeś/aś prawdziwy lab w chmurze!')
else:
    print('Nie wklejono jeszcze linka. Jeśli nie ukończyłeś/aś laba — nic się nie stało!')
    print('Możesz wrócić do tego zadania w dowolnym momencie.')

---

## Podsumowanie

Świetna robota! W tym laboratorium poznałeś/aś kluczowe usługi Google Cloud Platform dla Big Data:

### Co przećwiczyliśmy:

| # | Temat | Kluczowe umiejętności |
|---|-------|---------------------|
| 1 | Połączenie z BigQuery | `bigquery.Client()`, listowanie datasetów, pierwsze zapytanie |
| 2 | Eksploracja danych | `COUNT`, `DISTINCT`, `MIN`/`MAX` na dużych tabelach |
| 3 | Analiza SQL | `GROUP BY`, `ORDER BY`, `EXTRACT`, łączenie warunków |
| 4 | Wizualizacja | BigQuery → pandas DataFrame → matplotlib |
| 5 | Tworzenie klastra | `gcloud dataproc clusters create`, flagi konfiguracyjne |
| 6 | PySpark job | Skrypt `.py`, `gcloud dataproc jobs submit pyspark` |
| 7 | Optymalizacja kosztów | On-demand vs flat-rate, standard vs preemptible |

### Kluczowe wnioski:

1. **BigQuery** = serverless SQL na petabajtach danych — nie musisz zarządzać infrastrukturą
2. **Dataproc** = zarządzane klastry Spark — tworzysz w 90 sekund, usuwasz po zakończeniu
3. **Cloud Storage** = hub dla danych — łączy BigQuery, Dataproc i inne usługi
4. **Koszty** = ZAWSZE licz przed uruchomieniem! Preemptible workers mogą zaoszczędzić 50%+
5. **Bezpieczeństwo** = usuwaj klastry po zakończeniu, ustawiaj `--max-idle`

### Dalsza nauka:

- [BigQuery dokumentacja](https://cloud.google.com/bigquery/docs)
- [Dataproc dokumentacja](https://cloud.google.com/dataproc/docs)
- [Cloud Skills Boost](https://www.cloudskillsboost.google/) — darmowe laby
- [GCP Pricing Calculator](https://cloud.google.com/products/calculator)

---

> **Na następnych zajęciach:** Zaawansowane tematy chmurowe — Dataflow, Pub/Sub, serverless ML z Vertex AI!  
> **Do zobaczenia!**